# Boston 住宅価格予測データセットの探索と回帰モデルの訓練

このノートブックでは、Boston データセットを使った回帰問題を TDD で実装します。

## 1. 環境設定とパスの確認

In [9]:
import java.io.File

// 現在のワーキングディレクトリを確認
val currentDir = File(".").absolutePath
println("現在のディレクトリ: $currentDir")

// データファイルのパスを自動検出
val possiblePaths = listOf(
    "src/main/resources/data/Boston.csv",           // app/kotlin から実行
    "../src/main/resources/data/Boston.csv",        // notebook から実行
    "app/kotlin/src/main/resources/data/Boston.csv", // プロジェクトルートから実行
    "../../src/main/resources/data/Boston.csv"      // さらに深い場所から実行
)

val dataPath = possiblePaths.firstOrNull { File(it).exists() }
    ?: error("Boston.csv が見つかりません。現在のディレクトリ: $currentDir")

println("データファイル: $dataPath")
println("ファイル存在確認: ${File(dataPath).exists()}")

現在のディレクトリ: C:\Users\PC202411-1\IdeaProjects\case-study-game-dev\app\kotlin\notebook\.
データファイル: ../src/main/resources/data/Boston.csv
ファイル存在確認: true


## 2. 依存関係の設定

In [10]:
// Smile ML ライブラリと BLAS 実装の依存関係を追加
@file:Repository("https://jitpack.io")
@file:DependsOn("com.github.haifengl:smile-core:3.0.2")
@file:DependsOn("com.github.haifengl:smile-kotlin:3.0.2")
@file:DependsOn("org.bytedeco:openblas-platform:0.3.21-1.5.8")

println("依存関係の設定完了")

依存関係の設定完了


## 3. データの読み込みと概要確認

In [11]:
// CSV ファイルからデータを読み込む関数
fun loadBostonData(filePath: String): Pair<Array<DoubleArray>, DoubleArray> {
    val file = java.io.File(filePath)
    require(file.exists()) { "File not found: $filePath" }
    
    val lines = file.readLines()
    require(lines.isNotEmpty()) { "Empty file: $filePath" }
    
    val headerLine = lines[0].replace("\uFEFF", "").trim()
    val header = headerLine.split(",").map { it.trim() }
    
    val rmIdx = header.indexOfFirst { it.equals("RM", ignoreCase = true) }
    val lstatIdx = header.indexOfFirst { it.equals("LSTAT", ignoreCase = true) }
    val ptratioIdx = header.indexOfFirst { it.equals("PTRATIO", ignoreCase = true) }
    val priceIdx = header.indexOfFirst { it.equals("PRICE", ignoreCase = true) }
    
    require(rmIdx >= 0 && lstatIdx >= 0 && ptratioIdx >= 0 && priceIdx >= 0) {
        "Required columns not found in CSV"
    }
    
    val dataRows = mutableListOf<Pair<DoubleArray, Double>>()
    
    for (i in 1 until lines.size) {
        val values = lines[i].split(",")
        if (values.size != header.size) continue
        
        try {
            val rm = values[rmIdx].trim().toDoubleOrNull() ?: continue
            val lstat = values[lstatIdx].trim().toDoubleOrNull() ?: continue
            val ptratio = values[ptratioIdx].trim().toDoubleOrNull() ?: continue
            val price = values[priceIdx].trim().toDoubleOrNull() ?: continue
            
            val features = doubleArrayOf(rm, lstat, ptratio)
            dataRows.add(Pair(features, price))
        } catch (e: Exception) {
            continue
        }
    }
    
    val X = Array(dataRows.size) { dataRows[it].first }
    val y = DoubleArray(dataRows.size) { dataRows[it].second }
    
    return Pair(X, y)
}

// データの読み込み
val (X, y) = loadBostonData(dataPath)

println("=".repeat(60))
println("データの概要")
println("=".repeat(60))
println("サンプル数: ${X.size}")
println("特徴量数: ${X[0].size}")
println("特徴量: RM, LSTAT, PTRATIO")
println()

// データの最初の5行を表示
println("最初の5サンプル:")
println("RM, LSTAT, PTRATIO, PRICE")
for (i in 0 until minOf(5, X.size)) {
    println("%.2f, %.2f, %.2f, %.2f".format(X[i][0], X[i][1], X[i][2], y[i]))
}

データの概要
サンプル数: 100
特徴量数: 3
特徴量: RM, LSTAT, PTRATIO

最初の5サンプル:
RM, LSTAT, PTRATIO, PRICE
3.56, 7.12, 20.20, 27.50
5.95, 27.71, 21.00, 13.20
6.16, 7.43, 14.70, 24.10
6.15, 18.46, 21.20, 17.80
6.98, 11.66, 20.20, 29.80


## 4. Lets-Plot の初期化

In [12]:
%use lets-plot

## 5. データの可視化

### 5.1 価格（PRICE）の分布

In [13]:
// 価格（PRICE）の分布をヒストグラムで可視化
val priceValues = y.toList()

val p = letsPlot() + 
    geomHistogram(bins = 30) { x = priceValues }

p

5 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 2 
 
 
 
 
 
 
 4 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 12 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 x

### 5.2 部屋数（RM）の分布

In [14]:
// 部屋数（RM）の分布をヒストグラムで可視化
val rmValues = X.map { it[0] }

val p = letsPlot() + 
    geomHistogram(bins = 20) { x = rmValues }

p

4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 9 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 x

### 5.3 散布図 - 部屋数（RM）vs 価格（PRICE）

In [15]:
// 部屋数（RM）と価格（PRICE）の散布図
val rmData = X.map { it[0] }
val priceData = y.toList()

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.7) { 
        x = rmData
        y = priceData
    }

p

4 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 6 
 
 
 
 
 
 
 
 
 7 
 
 
 
 
 
 
 
 
 8 
 
 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

### 5.4 散布図 - 低所得層の割合（LSTAT）vs 価格（PRICE）

In [16]:
// 低所得層の割合（LSTAT）と価格（PRICE）の散布図
val lstatData = X.map { it[1] }

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.7) { 
        x = lstatData
        y = priceData
    }

p

5 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

### 5.5 散布図 - 生徒教師比（PTRATIO）vs 価格（PRICE）

In [17]:
// 生徒教師比（PTRATIO）と価格（PRICE）の散布図
val ptratioData = X.map { it[2] }

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.7) { 
        x = ptratioData
        y = priceData
    }

p

13 
 
 
 
 
 
 
 
 
 14 
 
 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 
 
 16 
 
 
 
 
 
 
 
 
 17 
 
 
 
 
 
 
 
 
 18 
 
 
 
 
 
 
 
 
 19 
 
 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 21 
 
 
 
 
 
 
 
 
 22 
 
 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 35 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 45 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

## 6. 基本統計量の確認

In [18]:
val featureNames = listOf("RM", "LSTAT", "PTRATIO")

println("特徴量の基本統計量:")
println("-".repeat(60))

featureNames.forEachIndexed { idx, name ->
    val values = X.map { it[idx] }
    val min = values.minOrNull() ?: 0.0
    val max = values.maxOrNull() ?: 0.0
    val mean = values.average()
    val sorted = values.sorted()
    val median = if (sorted.size % 2 == 0) {
        (sorted[sorted.size / 2 - 1] + sorted[sorted.size / 2]) / 2.0
    } else {
        sorted[sorted.size / 2]
    }
    
    println("%-10s: 最小=%.2f, 最大=%.2f, 平均=%.2f, 中央値=%.2f".format(name, min, max, mean, median))
}

println()
println("価格（PRICE）の基本統計量:")
println("-".repeat(60))
val priceMin = y.minOrNull() ?: 0.0
val priceMax = y.maxOrNull() ?: 0.0
val priceMean = y.average()
val priceSorted = y.sorted()
val priceMedian = if (priceSorted.size % 2 == 0) {
    (priceSorted[priceSorted.size / 2 - 1] + priceSorted[priceSorted.size / 2]) / 2.0
} else {
    priceSorted[priceSorted.size / 2]
}

println("PRICE:      最小=%.2f, 最大=%.2f, 平均=%.2f, 中央値=%.2f".format(
    priceMin, priceMax, priceMean, priceMedian
))

特徴量の基本統計量:
------------------------------------------------------------
RM        : 最小=3.56, 最大=8.70, 平均=6.24, 中央値=6.14
LSTAT     : 最小=1.92, 最大=30.59, 平均=11.83, 中央値=10.45
PTRATIO   : 最小=13.00, 最大=22.00, 平均=18.52, 中央値=18.65

価格（PRICE）の基本統計量:
------------------------------------------------------------
PRICE:      最小=5.00, 最大=50.00, 平均=23.46, 中央値=21.50


## 7. 相関分析

In [19]:
import kotlin.math.sqrt

// 相関係数を計算する関数
fun correlation(x: List<Double>, y: List<Double>): Double {
    require(x.size == y.size) { "Lists must have the same size" }
    val n = x.size
    val meanX = x.average()
    val meanY = y.average()
    
    var sumXY = 0.0
    var sumX2 = 0.0
    var sumY2 = 0.0
    
    for (i in 0 until n) {
        val dx = x[i] - meanX
        val dy = y[i] - meanY
        sumXY += dx * dy
        sumX2 += dx * dx
        sumY2 += dy * dy
    }
    
    return sumXY / sqrt(sumX2 * sumY2)
}

println("価格（PRICE）との相関係数:")
println("-".repeat(60))

val correlations = mutableListOf<Pair<String, Double>>()

featureNames.forEachIndexed { idx, feature ->
    val featureValues = X.map { it[idx] }
    val corr = correlation(featureValues, priceData)
    correlations.add(Pair(feature, corr))
    println("$feature: %.4f".format(corr))
}
println()

価格（PRICE）との相関係数:
------------------------------------------------------------
RM: 0.6867
LSTAT: -0.6851
PTRATIO: -0.4538



### 相関係数の棒グラフ

In [20]:
// 相関係数の棒グラフ
val featureLabels = correlations.map { it.first }
val corrValues = correlations.map { it.second }

val p = letsPlot() + 
    geomBar(stat = Stat.identity, alpha = 0.8) { 
        x = featureLabels
        y = corrValues
    }

p

RM 
 
 
 
 
 
 
 
 
 LSTAT 
 
 
 
 
 
 
 
 
 PTRATIO 
 
 
 
 
 
 
 
 
 
 
 -0.6 
 
 
 
 
 
 
 -0.4 
 
 
 
 
 
 
 -0.2 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

## まとめ

このノートブックでは、Boston データセットの基本的な探索と可視化を行いました。

### 実施内容

1. **データの読み込みと探索**: CSV からのデータ読み込み、基本統計量の確認
2. **データの可視化**: ヒストグラム、散布図による分布と相関の確認
3. **相関分析**: 各特徴量と価格の相関係数を計算

### 次のステップ

- データの前処理（CRIME ダミー変数化、外れ値除外）
- 特徴量エンジニアリング（2乗項、交互作用項、標準化）
- モデルの訓練と評価（OLS 回帰）
- 予測値と実測値の比較

---

お疲れ様でした！